# Static Move Prediction Draft

## Setup Environment && Load Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dataclasses import dataclass

import mlflow
import optuna

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

from chesswinnerprediction.static_move.models.static_move_base import StaticMoveBaseModel

from config import RANDOM_STATE
from chesswinnerprediction.static_move.utils import load_train_valid_test, setup_mlflow, log_prediction

## Setup MLflow

In [3]:
setup_mlflow(experiment_name="Static Move Prediction")

## Load Data

In [4]:
X_train, y_train, X_valid, y_valid, X_test, y_test = load_train_valid_test(drop_event=False)

train_sample_weight = X_train["sample_weight"]
X_train.drop(columns=["sample_weight"], inplace=True)

In [5]:
X_train.head()

,GameId,Event,WhiteElo,BlackElo,eval,EloDiff,MeanElo,BaseTime,IncrementTime,ZeroIncrementTime,white_remaining_time,black_remaining_time,mean_base_time,white_remaining_time_norm,black_remaining_time_norm,GameDurations,is_checkmate_countdown,i_move
0,341262,Rated Classical game,1981,1989,-16.04,-8,1985.0,600,3,0,301.0,23.0,737.970106,0.407876,0.031167,1221.0,False,117
1,62403,Rated Classical game,2066,2199,0.72,-133,2132.5,480,4,0,33.0,300.0,737.970106,0.044717,0.406521,987.0,False,92
2,134483,Rated Classical game,1933,1817,-8.72,116,1875.0,900,0,1,444.0,391.0,737.970106,0.601650,0.529832,965.0,False,113
3,898230,Rated Classical game,1408,1351,-1.86,57,1379.5,600,3,0,431.0,399.0,737.970106,0.584034,0.540672,499.0,False,45
4,468134,Rated Blitz game,1244,1329,-1.24,-85,1286.5,180,2,0,38.0,98.0,249.477434,0.152318,0.392821,378.0,False,79


## Optuna

In [11]:
def get_model(trial: optuna.Trial, model_type):
    if model_type == RandomForestClassifier.__name__:
        model = RandomForestClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
            n_estimators=trial.suggest_int("forest_n_estimators", 10, 250),
            max_depth=trial.suggest_int("forest_max_depth", 6, 20),
            min_samples_leaf=trial.suggest_int("forest_min_samples_leaf", 1, X_train.shape[0]//4),
            max_features="sqrt",
            ccp_alpha=trial.suggest_float("forest_ccp_alpha", 0.0, 0.1),
        )
    elif model_type == GradientBoostingClassifier.__name__:
        model = GradientBoostingClassifier(
            n_iter_no_change=5,
            tol=1e-3,
            random_state=RANDOM_STATE,
            learning_rate=trial.suggest_float("boosting_learning_rate", 1e-3, 1e-1, log=True),
            n_estimators=trial.suggest_int("boosting_n_estimators", 2, 150),
            max_depth=trial.suggest_int("boosting_max_depth", 2, 16),
            min_samples_leaf=trial.suggest_int(f"boosting_min_samples_leaf", 1, X_train.shape[0]//4),
            max_features="sqrt",
        )
    elif model_type == HistGradientBoostingClassifier.__name__:
        model = HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            tol=1e-5,
            n_iter_no_change=10,
            learning_rate=trial.suggest_float("hist_learning_rate", 1e-3, 1e-1, log=True),
            max_iter=trial.suggest_int("hist_max_iter", 100, 1000),
            max_depth=trial.suggest_int("hist_max_depth", 2, 16),
            min_samples_leaf=trial.suggest_int("hist_min_samples_leaf", 1, X_train.shape[0]//4),
            # max_leaf_nodes=trial.suggest_int("hist_max_leaf_nodes", 31, 255),
            # l2_regularization=trial.suggest_float("hist_l2_regularization", 1e-10, 1e-2, log=True),
            max_bins=trial.suggest_int("hist_max_bins", 128, 255),
            categorical_features=["Event"],
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    return model

In [12]:
@dataclass
class BestModel:
    trial: optuna.Trial = None
    score: float = 0.0
    run_id: str = ""
    model: StaticMoveBaseModel = None

In [13]:
class Objective:
    def __init__(self, parent_run_id, model_types, fit_kwargs):
        self.parent_run_id = parent_run_id
        self.model_types = model_types
        self.fit_kwargs = fit_kwargs
        self._best_models = self.__generate_best_model_dict()
        
    def __call__(self, trial: optuna.Trial):
        model_type = trial.suggest_categorical("model_type", self.model_types)

        estimator = get_model(trial, model_type)
        model = StaticMoveBaseModel(estimator=estimator)
        fit_kwargs = self.fit_kwargs[model_type]
        model.fit(X_train, y_train, **fit_kwargs)

        model.score(X_valid, y_valid)
        self.log_trial(trial, model)
        
        return model.balanced_accuracy
    
    def log_trial(self, trial, model: StaticMoveBaseModel):
        score = model.balanced_accuracy
        model_type = trial.params["model_type"]
        if score > self._best_models[model_type].score:
            self._best_models[model_type].model = model
            self._best_models[model_type].score = score
            self._best_models[model_type].trial = trial
        
        
        run_id = self._best_models[model_type].run_id
        with mlflow.start_run(nested=True, run_id=run_id, parent_run_id=self.parent_run_id) as model_run:
            with mlflow.start_run(nested=True, run_name=f"Trial_{trial.number}", parent_run_id=model_run.info.run_id):
                # mlflow.log_params(trial.params)
                model.log_trial()
        
    def log_best(self, trial):
        best_mode = self._best_models[trial.params["model_type"]]
        mlflow.log_params(best_mode.trial.params)
        best_mode.model.log_trial()
        
        for model_type in self.model_types:
            best_model_data = self._best_models[model_type]
            if best_model_data.model is None:
                mlflow.delete_run(best_model_data.run_id)
                continue
            with mlflow.start_run(nested=True, run_id=best_model_data.run_id, parent_run_id=self.parent_run_id) as model_run:
                mlflow.log_params(best_model_data.trial.params)
                best_model_data.model.log_trial()
                
    def log_prediction(self, trial, x, y, set_name):
        model = self._best_models[trial.params["model_type"]].model
        log_prediction(model, x, y, set_name)

          
    def __generate_best_model_dict(self):
        best_models = {}
        for model_type in self.model_types:
            with mlflow.start_run(nested=True, run_name=model_type, parent_run_id=self.parent_run_id) as model_run:
                best_models[model_type] = BestModel(run_id=model_run.info.run_id)
        return best_models

In [17]:
# models_types = [RandomForestClassifier.__name__, GradientBoostingClassifier.__name__]
models_types = [HistGradientBoostingClassifier.__name__]
models_fit_kwargs = {
    RandomForestClassifier.__name__: {},
    GradientBoostingClassifier.__name__: {},
    # HistGradientBoostingClassifier.__name__: {},
    HistGradientBoostingClassifier.__name__: {"sample_weight": train_sample_weight},
}

In [18]:
study_name = "Optuna: with sample_weight; new features"
# description = "study with sample_weight; no normed time features"
study = optuna.create_study(direction="maximize", study_name=study_name)

[I 2024-08-18 17:53:04,301] A new study created in memory with name: Optuna: with sample_weight; new features


In [19]:
with mlflow.start_run(run_name=study_name) as main_run:
    objective = Objective(parent_run_id=main_run.info.run_id, model_types=models_types, fit_kwargs=models_fit_kwargs)
    study.optimize(objective, n_trials=16, show_progress_bar=True)
    objective.log_best(study.best_trial)
    objective.log_prediction(study.best_trial, X_train, y_train, set_name="X_train")
    # objective.log_prediction(study.best_trial, X_valid, y_valid)
    objective.log_prediction(study.best_trial, X_test, y_test, set_name="X_test")

  0%|          | 0/16 [00:00<?, ?it/s]

[I 2024-08-18 17:53:12,706] Trial 0 finished with value: 0.5726 and parameters: {'model_type': 'HistGradientBoostingClassifier', 'hist_learning_rate': 0.09554109198735362, 'hist_max_iter': 362, 'hist_max_depth': 2, 'hist_min_samples_leaf': 6044, 'hist_max_bins': 217}. Best is trial 0 with value: 0.5726.
[I 2024-08-18 17:53:17,146] Trial 1 finished with value: 0.5369333333333334 and parameters: {'model_type': 'HistGradientBoostingClassifier', 'hist_learning_rate': 0.004536399170829462, 'hist_max_iter': 295, 'hist_max_depth': 9, 'hist_min_samples_leaf': 11218, 'hist_max_bins': 197}. Best is trial 0 with value: 0.5726.
[I 2024-08-18 17:53:34,271] Trial 2 finished with value: 0.5780666666666666 and parameters: {'model_type': 'HistGradientBoostingClassifier', 'hist_learning_rate': 0.005675298564629135, 'hist_max_iter': 751, 'hist_max_depth': 9, 'hist_min_samples_leaf': 1149, 'hist_max_bins': 250}. Best is trial 2 with value: 0.5780666666666666.
[I 2024-08-18 17:53:39,030] Trial 3 finished w

In [20]:
from sklearn.metrics import classification_report

In [21]:
best_model: HistGradientBoostingClassifier = get_model(study.best_trial, study.best_trial.params["model_type"])

In [22]:
best_model.verbose = 1

In [ ]:
best_model.fit(X_train, y_train)

In [24]:
y_pred = best_model.predict(X_test)

In [25]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0-1       0.61      0.64      0.62     10000
         1-0       0.61      0.67      0.64     10000
     1/2-1/2       0.67      0.58      0.62     10000

    accuracy                           0.63     30000
   macro avg       0.63      0.63      0.63     30000
weighted avg       0.63      0.63      0.63     30000



In [26]:
# show feature importance

In [27]:
from sklearn.inspection import permutation_importance


# Compute permutation importance
result = permutation_importance(best_model, X_train, y_train, n_repeats=10, random_state=42, n_jobs=-1)

# Display permutation importances
for i in result.importances_mean.argsort()[::-1]:
    print(f"Feature: {X_train.columns[i]}, Importance: {result.importances_mean[i]:.4f} +/- {result.importances_std[i]:.4f}")


Feature: eval, Importance: 0.1630 +/- 0.0018
Feature: i_move, Importance: 0.1585 +/- 0.0012
Feature: EloDiff, Importance: 0.0307 +/- 0.0008
Feature: white_remaining_time_norm, Importance: 0.0299 +/- 0.0005
Feature: black_remaining_time_norm, Importance: 0.0268 +/- 0.0007
Feature: is_checkmate_countdown, Importance: 0.0265 +/- 0.0005
Feature: black_remaining_time, Importance: 0.0260 +/- 0.0005
Feature: WhiteElo, Importance: 0.0235 +/- 0.0005
Feature: MeanElo, Importance: 0.0226 +/- 0.0010
Feature: white_remaining_time, Importance: 0.0216 +/- 0.0005
Feature: IncrementTime, Importance: 0.0203 +/- 0.0006
Feature: BlackElo, Importance: 0.0166 +/- 0.0006
Feature: GameDurations, Importance: 0.0156 +/- 0.0009
Feature: GameId, Importance: 0.0155 +/- 0.0003
Feature: BaseTime, Importance: 0.0073 +/- 0.0006
Feature: Event, Importance: 0.0055 +/- 0.0004
Feature: ZeroIncrementTime, Importance: 0.0005 +/- 0.0001
Feature: mean_base_time, Importance: 0.0002 +/- 0.0001
